# `DynamicSimulator` Demo

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_solid, energy, sim_utils, loads, benchmark, py_newton_optimizer
import tri_mesh_viewer

m = mesh.Mesh('../../misc/examples/meshes/bunny_coarse.msh', degree=1)
es = elastic_solid.ElasticSolid(m, energy.CommonNeoHookeanYoungPoisson(3, E=2, nu=0.4)) # Silicon material (Y= 2MPa, nu=0.4)

es.rho = 1e-4 # mass density in kg/mm^3
g = loads.Gravity(es, g=[0, -9.81, 0]) # gravitational acceleration in N/kg

In [ ]:
v = tri_mesh_viewer.Viewer(es, wireframe=True)
v.makeOpaque(color='#48B3FF')
v.show()

In [ ]:
import dynamic_simulator
ds = dynamic_simulator.DynamicSimulator(es, [g], useLumpedMass=True, dt=0.1)

In [ ]:
# Glue to the ground.
ds.fixedVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_Y, tol=4)

In [ ]:
ds.method = ds.method.ImplicitNewmark
ds.setPostTimestepCallback(v.updater(1))
ds.optimizer.options.verbose = 0

In [ ]:
benchmark.reset()
cr = ds.run(0, 10.0)
benchmark.report()